In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings 
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_google_genai import ChatGoogleGenerativeAI

/var/folders/cf/w8d5qyf95hbfw2kycgt8m1j80000gn/T/ipykernel_39327/2864071351.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [3]:
load_dotenv(dotenv_path='.env', override=True)

True

In [4]:
csv_path = ("Data/cleaned_dataset.csv")

In [5]:
df = pd.read_csv("Data/cleaned_dataset.csv")

In [6]:
df.head()

,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [8]:
#Skapar text för embeddings 
df["text"] = (
    "Track name: " + df["track_name"].astype(str) +
    ". Artist: " + df["artists"].astype(str) +
    ". Album: " + df["album_name"].astype(str) +
    ". Genre: " + df["track_genre"].astype(str) +
    ". Popularity: " + df["popularity"].astype(str) +
    ". Danceability: " + df["danceability"].astype(str) +
    ". Energy: " + df["energy"].astype(str) +
    ". Tempo: " + df["tempo"].astype(str)
)

df["text"].head()

0    Track name: Comedy. Artist: Gen Hoshino. Album...
1    Track name: Ghost - Acoustic. Artist: Ben Wood...
2    Track name: To Begin Again. Artist: Ingrid Mic...
3    Track name: Can't Help Falling In Love. Artist...
4    Track name: Hold On. Artist: Chord Overstreet....
Name: text, dtype: str

In [9]:
# Skapar en lista av Document-objekt där varje objekt innehåller texten från "text"-kolumnen 
documents = []
for text in df["text"]:
    documents.append(Document(page_content=text))

len(documents)

113423

In [10]:
documents[0]

Document(metadata={}, page_content='Track name: Comedy. Artist: Gen Hoshino. Album: Comedy. Genre: acoustic. Popularity: 73. Danceability: 0.676. Energy: 0.461. Tempo: 87.917')

In [11]:
# Skapar en mindre lista av Document-objekt 
small_documents = documents[:45]
len(small_documents)

45

In [12]:
#skapar vecktordatabasen 
vectorstore = Chroma.from_documents(
    collection_name="Spotify",
    documents=small_documents,
    embedding=embeddings,
    persist_directory="chroma_spotify_db"
)

vectorstore.persist()

/var/folders/cf/w8d5qyf95hbfw2kycgt8m1j80000gn/T/ipykernel_39327/956744714.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [13]:
retriever = vectorstore.as_retriever()

In [14]:
results = retriever.invoke("What are some popular acoustic songs?")

results

[Document(metadata={}, page_content='Track name: Ghost - Acoustic. Artist: Ben Woodward. Album: Ghost (Acoustic). Genre: acoustic. Popularity: 55. Danceability: 0.42. Energy: 0.166. Tempo: 77.489'),
 Document(metadata={}, page_content='Track name: Ghost - Acoustic. Artist: Ben Woodward. Album: Ghost (Acoustic). Genre: acoustic. Popularity: 55. Danceability: 0.42. Energy: 0.166. Tempo: 77.489'),
 Document(metadata={}, page_content='Track name: Ghost - Acoustic. Artist: Ben Woodward. Album: Ghost (Acoustic). Genre: acoustic. Popularity: 55. Danceability: 0.42. Energy: 0.166. Tempo: 77.489'),
 Document(metadata={}, page_content='Track name: Ghost - Acoustic. Artist: Ben Woodward. Album: Ghost (Acoustic). Genre: acoustic. Popularity: 55. Danceability: 0.42. Energy: 0.166. Tempo: 77.489')]

In [15]:
#skapa en prompt 
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    "You are a music expert. Based on the following retrieved information, answer the question: {context}\n\nQuestion: {question}"
)

In [16]:
question = "What are some popular acoustic songs?"

In [23]:
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0.7
)

In [24]:
response = llm.invoke(final_prompt)
print(response.content)

Based on the information provided, one popular acoustic song is:

*   **"Ghost - Acoustic" by Ben Woodward** (from the album "Ghost (Acoustic)"). This track has a popularity score of 55, indicating a moderate level of popularity.


In [18]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

/Users/alihanif/datakvalitet/Rag_app/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/cf/w8d5qyf95hbfw2kycgt8m1j80000gn/T/ipykernel_39327/3437173589.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/antigravity-preview-05-2026
models/

In [25]:
question = "What are some popular acoustic songs?"

docs = retriever.invoke(question)

context = "\n".join([doc.page_content for doc in docs])

final_prompt = prompt.format(
    context=context,
    question=question
)

response = llm.invoke(final_prompt)

print(response.content)

Based on the retrieved information, one popular acoustic song is "Ghost - Acoustic" by Ben Woodward. It has a popularity score of 55.
